In [17]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
import numpy as np

#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [2]:
    
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
        select * from CUN.MATRIZ_CALENDARIO
        WHERE TRY_CONVERT(DATE, FF_CLASE, 103) > CAST(GETDATE() AS DATE)
        and TRY_CONVERT(DATE, FI_CLASE, 103) < CAST(GETDATE() AS DATE)
        ''')
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        periodos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [3]:
periodos_unicos = periodos['PERIODO'].unique()
periodos_unicos = periodos['PERIODO'].dropna().astype(str).unique()
valores_unicos_periodos = ",".join(f"''{p}''" for p in periodos_unicos)

In [4]:
valores_unicos_periodos

"''25T06'',''25P06'',''25V06'',''2026A'',''26V01'',''26P01'',''26T01'',''26ES1'',''26ET1'',''26E01''"

In [5]:
    
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f'''
SELECT *
FROM OPENQUERY(MOODLE_NEW,'
select 
c.id as courseid,
(select cc2.name 
 from mdl_course_categories cc2 
 where (string_to_array(cc.path, ''/''))[4]::integer = cc2.id) as programa,
c.shortname,
c.fullname as curso,
c.visible as cursoactivo,
u.id as userid,
u.username as cedula,
u.firstname as nombre,
u.lastname as apellidos,
u.email,
cc.name,
to_timestamp(u.lastaccess) as ultimoaccesoplataforma,
to_timestamp(ul.timeaccess) as ultimoaccesoalcurso,
split_part(c.fullname, ''/'', 1) as Materia,
split_part(c.fullname, ''/'', 2) as grupo,
split_part(c.fullname, ''/'', 3) as bloque,
split_part(c.fullname, ''/'', 4) as periodo,
count(distinct cm.id)::text || ''/'' ||
count(distinct case 
    when cmc.completionstate in (1,2) then cm.id 
end)::text as progreso,
cc.path
from mdl_user u
left join mdl_role_assignments ra 
    on ra.userid = u.id and ra.roleid = 5
join mdl_context ct 
    on ra.contextid = ct.id and ct.contextlevel = 50
join mdl_course c 
    on c.id = ct.instanceid 
join mdl_course_categories cc 
    on cc.id = c.category
left join mdl_user_lastaccess ul 
    on ul.userid = u.id and c.id = ul.courseid 
join mdl_course_modules cm 
    on cm.course = c.id and cm.completion <> 0 
left join mdl_course_modules_completion cmc 
    on cmc.coursemoduleid = cm.id and cmc.userid = u.id 
where split_part(c.fullname, ''/'', 4) in ({valores_unicos_periodos})
group by 
    c.id,
    c.fullname,
    u.id,
    u.username,
    u.firstname,
    u.lastname,
    u.email,
    cc.name,
    ul.timeaccess,
    cc.path
') 
        ''')
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        experiencia = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [7]:
experiencia['periodo'].unique()

array(['26ES1', '25V06', '25T06', '25P06', '26T01', '26P01', '26V01',
       '2026A', '26E01', '26ET1'], dtype=object)

In [12]:
experiencia_periodos = experiencia['periodo'].unique().copy()

In [13]:
para_orden = experiencia['cedula'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"


In [14]:
experiencia_periodos

array(['26ES1', '25V06', '25T06', '25P06', '26T01', '26P01', '26V01',
       '2026A', '26E01', '26ET1'], dtype=object)

In [19]:
    
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f'''
            SELECT * FROM CUN_REPOSITORIO.cun.estadistica_academica WITH (NOLOCK)
 
        ''')
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        estadistica = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [20]:
estadistica

,TIP_IDENTIFICACION,NUM_IDENTIFICACION,NOM_TERCERO,SEG_NOMBRE,PRI_APELLIDO,SEG_APELLIDO,GEN_TERCERO,COD_PERIODO,COD_UNIDAD,NOM_UNIDAD,...,PRO_ACUMULADO,NIVEL_FORMACION,TEL_RESIDENCIA,MUNICIPIO_SEDE,AÑO,SEMESTRE,TEL_CELULAR,PENSUM_LIMPIO,EST_ALUMNO,CONCORDANCIA
0,CC,1004376919,OSMAR,YESID,GOMEZ,MIRANDA,M,2023A,TSM12,ADMINISTRACION DE EMPRESAS AGROINDUSTRIALES,...,4.44,TECNICO,None,Santa Marta,2023.0,9.0,3017643376,1044,Graduado,OK
1,CC,1004376919,OSMAR,YESID,GOMEZ,MIRANDA,M,2023B,TSM12,ADMINISTRACION DE EMPRESAS AGROINDUSTRIALES,...,4.44,TECNICO,None,Santa Marta,2023.0,9.0,3017643376,1044,Graduado,OK
2,CC,1004376919,OSMAR,YESID,GOMEZ,MIRANDA,M,21I05,IP007,EDUCACION CONTINUADA,...,4.40,Modulos,None,Santa Marta,2021.0,1.0,3017643376,IN001,Egresado,INGLES
3,CC,1004376919,OSMAR,YESID,GOMEZ,MIRANDA,M,22I02,IP007,EDUCACION CONTINUADA,...,4.45,Modulos,None,Santa Marta,2022.0,2.0,3017643376,IN001,Egresado,INGLES
4,CC,1004376919,OSMAR,YESID,GOMEZ,MIRANDA,M,22I04,IP007,EDUCACION CONTINUADA,...,4.33,Modulos,None,Santa Marta,2022.0,3.0,3017643376,IN001,Egresado,INGLES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1113070,CC,9994800,CESAR,AUGUSTO,RAMIREZ,PARADA,M,2015I,1TI04,NEGOCIOS INTERNACIONALES,...,4.14,TECNICO,3212617819,Ibague,2015.0,9.0,3212617819,8012,Graduado,OK
1113071,CC,9994800,CESAR,AUGUSTO,RAMIREZ,PARADA,M,2016A,1PI04,NEGOCIOS INTERNACIONALES,...,4.18,PROFESIONAL,3212617819,Ibague,2016.0,10.0,3212617819,8012,Graduado,OK
1113072,CE,999604,ANTHONIN,ZAQUIEL,ALVIREZ,HUIZA,M,25I05,IV001,EDUCACION CONTINUADA,...,4.80,Modulos,3232504881,Bogota,2025.0,1.0,3232504881,IN001,Activo,INGLES
1113073,CE,999604,ANTHONIN,ZAQUIEL,ALVIREZ,HUIZA,M,25V05,VT15G,INGENIERIA DE SISTEMAS,...,4.72,TECNICO,3232504881,Bogota,2025.0,1.0,3232504881,VG002,Activo,OK


In [21]:
experiencia

,courseid,programa,shortname,curso,cursoactivo,userid,cedula,nombre,apellidos,email,name,ultimoaccesoplataforma,ultimoaccesoalcurso,materia,grupo,bloque,periodo,progreso,path
0,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,1085,9431123,JAIRO,PALACIOS MARTINEZ,jairo.palacios@cun.edu.co,1,2026-03-14 10:51:18.0000000,None,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557
1,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,13353,52366329,DURLEY\t COLOMBIA,AGUIRRE\t TORRES,durley.aguirre@cun.edu.co,1,2026-03-16 10:07:22.0000000,None,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557
2,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24025,1024522663,VICTOR ANDRES,LANCHEROS ARIAS,victor_lancheros@cun.edu.co,1,2026-03-16 13:48:31.0000000,None,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557
3,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24674,80108393,JUAN CARLOS,SIERRA PINEDA,juan.sierrappi@cun.edu.co,1,2026-03-15 16:20:13.0000000,None,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557
4,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,40716,51963453,ANA CRISTINA,TRIANA TOCANCIPA,ana.trianat@cun.edu.co,1,2026-03-15 09:14:54.0000000,None,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403677,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90414,1117516017,CINDY VANESSA,CARVAJAL ALVAREZ,cindy.carvajal@cun.edu.co,3,2026-03-16 07:30:23.0000000,None,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ,85ET1,SEGUNDO BLOQUE,26ET1,31/0,/803/37854/37863/37866/37867
403678,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90471,1012383887,DIEGO GERMAN,RODRIGUEZ GOMEZ,diego.rodriguezgommez@cun.edu.co,3,2026-03-16 07:30:38.0000000,None,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ,85ET1,SEGUNDO BLOQUE,26ET1,31/0,/803/37854/37863/37866/37867
403679,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,103678,1010232220,JAZMÍN ASTRID,MOYANO ORDÓÑEZ,jazmin.moyano@cun.edu.co,3,2026-03-12 21:05:02.0000000,None,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ,85ET1,SEGUNDO BLOQUE,26ET1,31/0,/803/37854/37863/37866/37867
403680,106699,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,24084/50101/2026A/B2/1PC11/3028N,NARRATIVA AUDIOVISUAL/50101/SEGUNDO BLOQUE/2026A,0,54573,1016832087,SAMUEL DAVID,HERRERA UBATE,samuel.herrerau@cun.edu.co,3,2026-03-02 18:55:42.0000000,None,NARRATIVA AUDIOVISUAL,50101,SEGUNDO BLOQUE,2026A,37/0,/784/34256/36198/38010/38016


In [22]:
estadistica_unica = estadistica.drop_duplicates(
    subset=['NUM_IDENTIFICACION', 'COD_PERIODO']
)

experiencia = experiencia.merge(
    estadistica_unica[['NUM_IDENTIFICACION', 'COD_PERIODO',
                       'NOM_UNIDAD', 'MODALIDAD', 'EST_ALUMNO', 'COD_UNIDAD']],
    left_on=['cedula', 'periodo'],
    right_on=['NUM_IDENTIFICACION', 'COD_PERIODO'],
    how='left'
).drop(columns=['NUM_IDENTIFICACION', 'COD_PERIODO'])

In [23]:
experiencia

,courseid,programa,shortname,curso,cursoactivo,userid,cedula,nombre,apellidos,email,...,materia,grupo,bloque,periodo,progreso,path,NOM_UNIDAD,MODALIDAD,EST_ALUMNO,COD_UNIDAD
0,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,1085,9431123,JAIRO,PALACIOS MARTINEZ,jairo.palacios@cun.edu.co,...,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,VIRTUAL,None,ESP22
1,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,13353,52366329,DURLEY\t COLOMBIA,AGUIRRE\t TORRES,durley.aguirre@cun.edu.co,...,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,VIRTUAL,None,ESP22
2,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24025,1024522663,VICTOR ANDRES,LANCHEROS ARIAS,victor_lancheros@cun.edu.co,...,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,VIRTUAL,None,ESP22
3,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24674,80108393,JUAN CARLOS,SIERRA PINEDA,juan.sierrappi@cun.edu.co,...,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,VIRTUAL,None,ESP22
4,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,40716,51963453,ANA CRISTINA,TRIANA TOCANCIPA,ana.trianat@cun.edu.co,...,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,VIRTUAL,None,ESP22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403677,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90414,1117516017,CINDY VANESSA,CARVAJAL ALVAREZ,cindy.carvajal@cun.edu.co,...,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ,85ET1,SEGUNDO BLOQUE,26ET1,31/0,/803/37854/37863/37866/37867,NaN,NaN,NaN,NaN
403678,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90471,1012383887,DIEGO GERMAN,RODRIGUEZ GOMEZ,diego.rodriguezgommez@cun.edu.co,...,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ,85ET1,SEGUNDO BLOQUE,26ET1,31/0,/803/37854/37863/37866/37867,NaN,NaN,NaN,NaN
403679,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,103678,1010232220,JAZMÍN ASTRID,MOYANO ORDÓÑEZ,jazmin.moyano@cun.edu.co,...,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ,85ET1,SEGUNDO BLOQUE,26ET1,31/0,/803/37854/37863/37866/37867,NaN,NaN,NaN,NaN
403680,106699,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,24084/50101/2026A/B2/1PC11/3028N,NARRATIVA AUDIOVISUAL/50101/SEGUNDO BLOQUE/2026A,0,54573,1016832087,SAMUEL DAVID,HERRERA UBATE,samuel.herrerau@cun.edu.co,...,NARRATIVA AUDIOVISUAL,50101,SEGUNDO BLOQUE,2026A,37/0,/784/34256/36198/38010/38016,DIRECCION Y PRODUCCION DE MEDIOS AUDIOVISUALES,PRESENCIAL,None,1TC11


In [24]:
    
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f'''
select SEMESTRE,COD_PERIODO,NUM_IDENTIFICACION,DIR_EMAIL_PER,DIR_EMAIL,TEL_CELULAR,NOM_SECCIONAL,cod_pensum,
G.DEPARTAMENTO AS DEPARTAMENTO_RESIDENCIA, G.CIUDAD AS CIUDAD_RESIDENCIA,G.localidad_normalizada
from CUN_REPOSITORIO.cun.estadistica_estudiante_2 E WITH (NOLOCK)
LEFT JOIN CUN_REPOSITORIO.GEO.estadistica_geolocalizacion G WITH (NOLOCK)
ON E.NUM_IDENTIFICACION = G.Identificacion
        ''')
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        estadisticaES = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [25]:
estadisticaES

,SEMESTRE,COD_PERIODO,NUM_IDENTIFICACION,DIR_EMAIL_PER,DIR_EMAIL,TEL_CELULAR,NOM_SECCIONAL,cod_pensum,DEPARTAMENTO_RESIDENCIA,CIUDAD_RESIDENCIA,localidad_normalizada
0,3,23I04,1085934490,manuel.castrop@cun.edu.co,manuel.castrop@cun.edu.co,+57 3185339832,Regional Bogotá,IN001,NARIÑO,IPIALES,Sin localidad
1,2,23I04,1143128653,manuel.blandon@cun.edu.co,manuel.blandon@cun.edu.co,+573045346750,Regional Bogotá,IN001,ANTIOQUIA,MEDELLIN,Aranjuez
2,1,23I04,1025525016,dh1525cruz@gmail.com,manuel.hurtadod@cun.edu.co,3002186930,Regional Bogotá,IN001,VALLE DEL CAUCA,CALI,Comuna 14
3,1,23I04,1042774115,a.veronik16@gmail.com,veronica.piedrahita@cun.edu.co,3117574554,Regional Bogotá,IN001,ANTIOQUIA,YARUMAL,Barrio Señor Caido
4,1,23I04,1003265365,veronica.floreza@cun.edu.co,veronica.floreza@cun.edu.co,3147780871,Regional Bogotá,IN001,DISTRITO CAPITAL,BOGOTA,Suba
...,...,...,...,...,...,...,...,...,...,...,...
1216859,7,21V02,10953260,slcguerra@hotmail.com,juan.guerra@cun.edu.co,+57 3218289917,Regional Bogotá,1037,CORDOBA,MONTERIA,Comuna 4: El Recreo
1216860,7,21V02,1120563973,juankavieres@gmail.com,juan.hernandezcav@cun.edu.co,3166436238,Regional Bogotá,1037,CALDAS,SAN JOSE,Sin localidad
1216861,1,21V02,79905667,juanpedrazaq@gmail.com,juan.pedrazaq@cun.edu.co,573203912111,Regional Bogotá,1110,DISTRITO CAPITAL,BOGOTA,Fontibón
1216862,1,21V02,1022444561,juanchis53@outlook.com,juan.rodriguezcca@cun.edu.co,+573214146313,Regional Bogotá,1120,DISTRITO CAPITAL,BOGOTA,Kennedy


In [26]:
experiencia

,courseid,programa,shortname,curso,cursoactivo,userid,cedula,nombre,apellidos,email,...,materia,grupo,bloque,periodo,progreso,path,NOM_UNIDAD,MODALIDAD,EST_ALUMNO,COD_UNIDAD
0,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,1085,9431123,JAIRO,PALACIOS MARTINEZ,jairo.palacios@cun.edu.co,...,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,VIRTUAL,None,ESP22
1,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,13353,52366329,DURLEY\t COLOMBIA,AGUIRRE\t TORRES,durley.aguirre@cun.edu.co,...,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,VIRTUAL,None,ESP22
2,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24025,1024522663,VICTOR ANDRES,LANCHEROS ARIAS,victor_lancheros@cun.edu.co,...,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,VIRTUAL,None,ESP22
3,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24674,80108393,JUAN CARLOS,SIERRA PINEDA,juan.sierrappi@cun.edu.co,...,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,VIRTUAL,None,ESP22
4,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,40716,51963453,ANA CRISTINA,TRIANA TOCANCIPA,ana.trianat@cun.edu.co,...,GERENCIA SOCIAL,51E2T,SEGUNDO BLOQUE,26ES1,32/0,/803/31554/31555/31556/31557,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,VIRTUAL,None,ESP22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403677,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90414,1117516017,CINDY VANESSA,CARVAJAL ALVAREZ,cindy.carvajal@cun.edu.co,...,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ,85ET1,SEGUNDO BLOQUE,26ET1,31/0,/803/37854/37863/37866/37867,NaN,NaN,NaN,NaN
403678,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90471,1012383887,DIEGO GERMAN,RODRIGUEZ GOMEZ,diego.rodriguezgommez@cun.edu.co,...,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ,85ET1,SEGUNDO BLOQUE,26ET1,31/0,/803/37854/37863/37866/37867,NaN,NaN,NaN,NaN
403679,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,103678,1010232220,JAZMÍN ASTRID,MOYANO ORDÓÑEZ,jazmin.moyano@cun.edu.co,...,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ,85ET1,SEGUNDO BLOQUE,26ET1,31/0,/803/37854/37863/37866/37867,NaN,NaN,NaN,NaN
403680,106699,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,24084/50101/2026A/B2/1PC11/3028N,NARRATIVA AUDIOVISUAL/50101/SEGUNDO BLOQUE/2026A,0,54573,1016832087,SAMUEL DAVID,HERRERA UBATE,samuel.herrerau@cun.edu.co,...,NARRATIVA AUDIOVISUAL,50101,SEGUNDO BLOQUE,2026A,37/0,/784/34256/36198/38010/38016,DIRECCION Y PRODUCCION DE MEDIOS AUDIOVISUALES,PRESENCIAL,None,1TC11


In [27]:
estadistica_unicaES = estadisticaES.drop_duplicates(
    subset=['NUM_IDENTIFICACION', 'COD_PERIODO']
)

experienciaf = experiencia.merge(
    estadistica_unicaES[
        ['NUM_IDENTIFICACION','SEMESTRE', 'COD_PERIODO','cod_pensum',
         'DIR_EMAIL_PER', 'DIR_EMAIL','TEL_CELULAR',
         'NOM_SECCIONAL', 'DEPARTAMENTO_RESIDENCIA',
         'CIUDAD_RESIDENCIA','localidad_normalizada']
    ],
    left_on=['cedula', 'periodo'],
    right_on=['NUM_IDENTIFICACION', 'COD_PERIODO'],
    how='left',
    validate='many_to_one'   # 🔥 recomendado
).drop(columns=['NUM_IDENTIFICACION', 'COD_PERIODO'])

In [28]:
experienciaf

,courseid,programa,shortname,curso,cursoactivo,userid,cedula,nombre,apellidos,email,...,COD_UNIDAD,SEMESTRE,cod_pensum,DIR_EMAIL_PER,DIR_EMAIL,TEL_CELULAR,NOM_SECCIONAL,DEPARTAMENTO_RESIDENCIA,CIUDAD_RESIDENCIA,localidad_normalizada
0,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,1085,9431123,JAIRO,PALACIOS MARTINEZ,jairo.palacios@cun.edu.co,...,ESP22,1,ESPPA,amigosdeyopal82@hotmail.com,jairo.palacios@cun.edu.co,3209250213,Regional Bogotá,CASANARE,YOPAL,Calixto Zambrano
1,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,13353,52366329,DURLEY\t COLOMBIA,AGUIRRE\t TORRES,durley.aguirre@cun.edu.co,...,ESP22,1,ESPPA,dcaguirret@gmail.com,durley.aguirre@cun.edu.co,3104682485,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Ciudad Bolívar
2,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24025,1024522663,VICTOR ANDRES,LANCHEROS ARIAS,victor_lancheros@cun.edu.co,...,ESP22,2,ESPPA,victor_lancheros@cun.edu.co,victor_lancheros@cun.edu.co,3195641604,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Bosa
3,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24674,80108393,JUAN CARLOS,SIERRA PINEDA,juan.sierrappi@cun.edu.co,...,ESP22,1,ESPPA,juan.sierra@correo.policia.gov.co,juan.sierrappi@cun.edu.co,3115892254,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Ciudad Bolívar
4,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,40716,51963453,ANA CRISTINA,TRIANA TOCANCIPA,ana.trianat@cun.edu.co,...,ESP22,2,ESPPA,anacristina18@live.com,ana.trianat@cun.edu.co,3144476485,Regional Bogotá,CASANARE,YOPAL,Ciudad Campiña
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403677,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90414,1117516017,CINDY VANESSA,CARVAJAL ALVAREZ,cindy.carvajal@cun.edu.co,...,NaN,4,ESPPA,cindycarvajal13@gmail.com,cindy.carvajal@cun.edu.co,3184022274,Regional Bogotá,CAQUETA,FLORENCIA,Comuna Suroriental
403678,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90471,1012383887,DIEGO GERMAN,RODRIGUEZ GOMEZ,diego.rodriguezgommez@cun.edu.co,...,NaN,4,ESPPA,digrodriguezgo@unal.edu.co,diego.rodriguezgommez@cun.edu.co,3208394302,Regional Bogotá,CUNDINAMARCA,SOACHA,Corregimiento 2 Norte
403679,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,103678,1010232220,JAZMÍN ASTRID,MOYANO ORDÓÑEZ,jazmin.moyano@cun.edu.co,...,NaN,4,ESPPA,jazmina.moyanoo@gmail.com,jazmin.moyano@cun.edu.co,3116785140,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Santa Fe
403680,106699,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,24084/50101/2026A/B2/1PC11/3028N,NARRATIVA AUDIOVISUAL/50101/SEGUNDO BLOQUE/2026A,0,54573,1016832087,SAMUEL DAVID,HERRERA UBATE,samuel.herrerau@cun.edu.co,...,1TC11,3,3028N,samuherrera112020@gmail.com,samuel.herrerau@cun.edu.co,3058946748,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Ciudad Bolívar


In [29]:
import pandas as pd
import numpy as np

# --------------------------
# 1️⃣ Convertir a datetime
# --------------------------
experienciaf['ultimoaccesoplataforma'] = pd.to_datetime(
    experienciaf['ultimoaccesoplataforma'], errors='coerce'
)

experienciaf['ultimoaccesoalcurso'] = pd.to_datetime(
    experienciaf['ultimoaccesoalcurso'], errors='coerce'
)

# --------------------------
# 2️⃣ Fecha actual sin hora
# --------------------------
hoy = pd.Timestamp.now().normalize()

# --------------------------
# 3️⃣ Calcular días sin ingreso
# --------------------------
# Plataforma
dias_plataforma = (hoy - experienciaf['ultimoaccesoplataforma'].dt.normalize()).dt.days
dias_plataforma = dias_plataforma.clip(lower=0)
experienciaf['ultimoaccesoplataformdias'] = dias_plataforma.fillna('No ingreso')

# Curso
dias_curso = (hoy - experienciaf['ultimoaccesoalcurso'].dt.normalize()).dt.days
dias_curso = dias_curso.clip(lower=0)
experienciaf['ultimoaccesoalcursodias'] = dias_curso.fillna('No ingreso')

# --------------------------
# 4️⃣ Crear columnas limpias en formato YYYY-MM-DD HH:MM:SS
# --------------------------
experienciaf['ultimoaccesoplataformlimpio'] = experienciaf['ultimoaccesoplataforma'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna('No ingreso')
experienciaf['ultimoaccesoalcursolimpio'] = experienciaf['ultimoaccesoalcurso'].dt.strftime('%Y-%m-%d %H:%M:%S').fillna('No ingreso')

In [30]:
experienciaf

,courseid,programa,shortname,curso,cursoactivo,userid,cedula,nombre,apellidos,email,...,DIR_EMAIL,TEL_CELULAR,NOM_SECCIONAL,DEPARTAMENTO_RESIDENCIA,CIUDAD_RESIDENCIA,localidad_normalizada,ultimoaccesoplataformdias,ultimoaccesoalcursodias,ultimoaccesoplataformlimpio,ultimoaccesoalcursolimpio
0,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,1085,9431123,JAIRO,PALACIOS MARTINEZ,jairo.palacios@cun.edu.co,...,jairo.palacios@cun.edu.co,3209250213,Regional Bogotá,CASANARE,YOPAL,Calixto Zambrano,2,No ingreso,2026-03-14 10:51:18,No ingreso
1,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,13353,52366329,DURLEY\t COLOMBIA,AGUIRRE\t TORRES,durley.aguirre@cun.edu.co,...,durley.aguirre@cun.edu.co,3104682485,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Ciudad Bolívar,0,No ingreso,2026-03-16 10:07:22,No ingreso
2,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24025,1024522663,VICTOR ANDRES,LANCHEROS ARIAS,victor_lancheros@cun.edu.co,...,victor_lancheros@cun.edu.co,3195641604,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Bosa,0,No ingreso,2026-03-16 13:48:31,No ingreso
3,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24674,80108393,JUAN CARLOS,SIERRA PINEDA,juan.sierrappi@cun.edu.co,...,juan.sierrappi@cun.edu.co,3115892254,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Ciudad Bolívar,1,No ingreso,2026-03-15 16:20:13,No ingreso
4,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,40716,51963453,ANA CRISTINA,TRIANA TOCANCIPA,ana.trianat@cun.edu.co,...,ana.trianat@cun.edu.co,3144476485,Regional Bogotá,CASANARE,YOPAL,Ciudad Campiña,1,No ingreso,2026-03-15 09:14:54,No ingreso
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403677,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90414,1117516017,CINDY VANESSA,CARVAJAL ALVAREZ,cindy.carvajal@cun.edu.co,...,cindy.carvajal@cun.edu.co,3184022274,Regional Bogotá,CAQUETA,FLORENCIA,Comuna Suroriental,0,No ingreso,2026-03-16 07:30:23,No ingreso
403678,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90471,1012383887,DIEGO GERMAN,RODRIGUEZ GOMEZ,diego.rodriguezgommez@cun.edu.co,...,diego.rodriguezgommez@cun.edu.co,3208394302,Regional Bogotá,CUNDINAMARCA,SOACHA,Corregimiento 2 Norte,0,No ingreso,2026-03-16 07:30:38,No ingreso
403679,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,103678,1010232220,JAZMÍN ASTRID,MOYANO ORDÓÑEZ,jazmin.moyano@cun.edu.co,...,jazmin.moyano@cun.edu.co,3116785140,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Santa Fe,4,No ingreso,2026-03-12 21:05:02,No ingreso
403680,106699,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,24084/50101/2026A/B2/1PC11/3028N,NARRATIVA AUDIOVISUAL/50101/SEGUNDO BLOQUE/2026A,0,54573,1016832087,SAMUEL DAVID,HERRERA UBATE,samuel.herrerau@cun.edu.co,...,samuel.herrerau@cun.edu.co,3058946748,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Ciudad Bolívar,14,No ingreso,2026-03-02 18:55:42,No ingreso


In [31]:
experienciafinal = experienciaf.drop(columns=['ultimoaccesoplataforma'])

In [32]:
experienciafinal

,courseid,programa,shortname,curso,cursoactivo,userid,cedula,nombre,apellidos,email,...,DIR_EMAIL,TEL_CELULAR,NOM_SECCIONAL,DEPARTAMENTO_RESIDENCIA,CIUDAD_RESIDENCIA,localidad_normalizada,ultimoaccesoplataformdias,ultimoaccesoalcursodias,ultimoaccesoplataformlimpio,ultimoaccesoalcursolimpio
0,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,1085,9431123,JAIRO,PALACIOS MARTINEZ,jairo.palacios@cun.edu.co,...,jairo.palacios@cun.edu.co,3209250213,Regional Bogotá,CASANARE,YOPAL,Calixto Zambrano,2,No ingreso,2026-03-14 10:51:18,No ingreso
1,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,13353,52366329,DURLEY\t COLOMBIA,AGUIRRE\t TORRES,durley.aguirre@cun.edu.co,...,durley.aguirre@cun.edu.co,3104682485,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Ciudad Bolívar,0,No ingreso,2026-03-16 10:07:22,No ingreso
2,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24025,1024522663,VICTOR ANDRES,LANCHEROS ARIAS,victor_lancheros@cun.edu.co,...,victor_lancheros@cun.edu.co,3195641604,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Bosa,0,No ingreso,2026-03-16 13:48:31,No ingreso
3,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24674,80108393,JUAN CARLOS,SIERRA PINEDA,juan.sierrappi@cun.edu.co,...,juan.sierrappi@cun.edu.co,3115892254,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Ciudad Bolívar,1,No ingreso,2026-03-15 16:20:13,No ingreso
4,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,40716,51963453,ANA CRISTINA,TRIANA TOCANCIPA,ana.trianat@cun.edu.co,...,ana.trianat@cun.edu.co,3144476485,Regional Bogotá,CASANARE,YOPAL,Ciudad Campiña,1,No ingreso,2026-03-15 09:14:54,No ingreso
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403677,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90414,1117516017,CINDY VANESSA,CARVAJAL ALVAREZ,cindy.carvajal@cun.edu.co,...,cindy.carvajal@cun.edu.co,3184022274,Regional Bogotá,CAQUETA,FLORENCIA,Comuna Suroriental,0,No ingreso,2026-03-16 07:30:23,No ingreso
403678,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90471,1012383887,DIEGO GERMAN,RODRIGUEZ GOMEZ,diego.rodriguezgommez@cun.edu.co,...,diego.rodriguezgommez@cun.edu.co,3208394302,Regional Bogotá,CUNDINAMARCA,SOACHA,Corregimiento 2 Norte,0,No ingreso,2026-03-16 07:30:38,No ingreso
403679,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,103678,1010232220,JAZMÍN ASTRID,MOYANO ORDÓÑEZ,jazmin.moyano@cun.edu.co,...,jazmin.moyano@cun.edu.co,3116785140,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Santa Fe,4,No ingreso,2026-03-12 21:05:02,No ingreso
403680,106699,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,24084/50101/2026A/B2/1PC11/3028N,NARRATIVA AUDIOVISUAL/50101/SEGUNDO BLOQUE/2026A,0,54573,1016832087,SAMUEL DAVID,HERRERA UBATE,samuel.herrerau@cun.edu.co,...,samuel.herrerau@cun.edu.co,3058946748,Regional Bogotá,DISTRITO CAPITAL,BOGOTA,Ciudad Bolívar,14,No ingreso,2026-03-02 18:55:42,No ingreso


In [33]:
    
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f'''
SELECT DOC_ALUM,PERIODO,COD_UNI,SEDE,TIPO,NUEVO,EST_MATRICULADO, NOM_PROGRAMA, LAT, LON FROM CUN_REPOSITORIO.ZOHO.BASE_PERSONAS WITH (NOLOCK)
                         where anio >= '2025'
        ''')
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        basepersona = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [34]:
basepersona.columns

Index(['DOC_ALUM', 'PERIODO', 'COD_UNI', 'SEDE', 'TIPO', 'NUEVO',
       'EST_MATRICULADO', 'NOM_PROGRAMA', 'LAT', 'LON'],
      dtype='object')

In [35]:
experienciafinal.columns

Index(['courseid', 'programa', 'shortname', 'curso', 'cursoactivo', 'userid',
       'cedula', 'nombre', 'apellidos', 'email', 'name', 'ultimoaccesoalcurso',
       'materia', 'grupo', 'bloque', 'periodo', 'progreso', 'path',
       'NOM_UNIDAD', 'MODALIDAD', 'EST_ALUMNO', 'COD_UNIDAD', 'SEMESTRE',
       'cod_pensum', 'DIR_EMAIL_PER', 'DIR_EMAIL', 'TEL_CELULAR',
       'NOM_SECCIONAL', 'DEPARTAMENTO_RESIDENCIA', 'CIUDAD_RESIDENCIA',
       'localidad_normalizada', 'ultimoaccesoplataformdias',
       'ultimoaccesoalcursodias', 'ultimoaccesoplataformlimpio',
       'ultimoaccesoalcursolimpio'],
      dtype='object')

In [36]:
basepersona = basepersona.drop_duplicates(
    subset=['DOC_ALUM', 'PERIODO']
)

experienciafinalv2 = experienciafinal.merge(
    basepersona[
        ['DOC_ALUM', 'PERIODO', 'COD_UNI',
         'SEDE', 'TIPO','NUEVO',
         'EST_MATRICULADO','NOM_PROGRAMA', 'LAT', 'LON']
    ],
    left_on=['cedula', 'periodo','COD_UNIDAD'],
    right_on=['DOC_ALUM', 'PERIODO','COD_UNI'],
    how='left',
    validate='many_to_one'   # 🔥 recomendado
).drop(columns=['DOC_ALUM', 'PERIODO'])

In [37]:
experienciafinalv2

,courseid,programa,shortname,curso,cursoactivo,userid,cedula,nombre,apellidos,email,...,ultimoaccesoplataformlimpio,ultimoaccesoalcursolimpio,COD_UNI,SEDE,TIPO,NUEVO,EST_MATRICULADO,NOM_PROGRAMA,LAT,LON
0,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,1085,9431123,JAIRO,PALACIOS MARTINEZ,jairo.palacios@cun.edu.co,...,2026-03-14 10:51:18,No ingreso,ESP22,BOGOTA CENTRO,Postgrado,NUEVO,1-Activo,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,5.34766,-72.3928
1,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,13353,52366329,DURLEY\t COLOMBIA,AGUIRRE\t TORRES,durley.aguirre@cun.edu.co,...,2026-03-16 10:07:22,No ingreso,ESP22,BOGOTA CENTRO,Postgrado,NUEVO,1-Activo,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,4.58652,-74.1756
2,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24025,1024522663,VICTOR ANDRES,LANCHEROS ARIAS,victor_lancheros@cun.edu.co,...,2026-03-16 13:48:31,No ingreso,ESP22,BOGOTA CENTRO,Postgrado,ANTIGUO,1-Activo,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,4.60076,-74.1661
3,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,24674,80108393,JUAN CARLOS,SIERRA PINEDA,juan.sierrappi@cun.edu.co,...,2026-03-15 16:20:13,No ingreso,ESP22,BOGOTA CENTRO,Postgrado,NUEVO,1-Activo,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,4.53905,-74.1551
4,83555,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP193/51E2T/26ES1/B2/ESP22/ESPPA,GERENCIA SOCIAL/51E2T/SEGUNDO BLOQUE/26ES1,0,40716,51963453,ANA CRISTINA,TRIANA TOCANCIPA,ana.trianat@cun.edu.co,...,2026-03-15 09:14:54,No ingreso,ESP22,BOGOTA CENTRO,Postgrado,ANTIGUO,1-Activo,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,5.33402,-72.3958
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403677,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90414,1117516017,CINDY VANESSA,CARVAJAL ALVAREZ,cindy.carvajal@cun.edu.co,...,2026-03-16 07:30:23,No ingreso,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
403678,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,90471,1012383887,DIEGO GERMAN,RODRIGUEZ GOMEZ,diego.rodriguezgommez@cun.edu.co,...,2026-03-16 07:30:38,No ingreso,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
403679,106674,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,ESP08/85ET1/26ET1/B2/ESP08/ESPPA,DESARROLLO TERRITORIAL Y COMPONENTES DE PAZ/85...,0,103678,1010232220,JAZMÍN ASTRID,MOYANO ORDÓÑEZ,jazmin.moyano@cun.edu.co,...,2026-03-12 21:05:02,No ingreso,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
403680,106699,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,24084/50101/2026A/B2/1PC11/3028N,NARRATIVA AUDIOVISUAL/50101/SEGUNDO BLOQUE/2026A,0,54573,1016832087,SAMUEL DAVID,HERRERA UBATE,samuel.herrerau@cun.edu.co,...,2026-03-02 18:55:42,No ingreso,1TC11,BOGOTA CENTRO,Pregrado,ANTIGUO,1-Activo,DIRECCION Y PRODUCCION DE MEDIOS AUDIOVISUALES,4.53116,-74.1524


In [39]:
periodos_unicos_2 = periodos['PERIODO'].unique()
valores_unicos_periodos_2 = "', '".join(map(str, periodos_unicos_2))
valores_unicos_periodos_2 = f"'{valores_unicos_periodos_2}'"

# ELIMINA LOS REGISTROS DE LOS PERIODOS ACTIVOS

In [41]:

with engine.begin() as connection:

    delete_query = text(f"""
        DELETE FROM DBARON.CURSOS_MOODLE_2026
        WHERE PERIODO IN ({valores_unicos_periodos_2})
    """)

    connection.execute(delete_query)

# CARGA LOS NUEVOS REGISTROS DE LOS PERIODOS ACTIVOS


In [42]:
# # Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'CURSOS_MOODLE_2026'
schema_name = 'DBARON'
# Send the data to sql server
experienciafinalv2.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()